# 1.1 SysEngBench: Download and Analysis

This notebook:
1. Downloads the SysEngBench dataset from HuggingFace
2. Exports the raw data to CSV
3. Transforms the data by exploding INCOSE Handbook categories
4. Provides statistical analysis of question distribution
5. Visualizes the benchmark composition

## Setup: Load HuggingFace Token

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("HF_TOKEN loaded successfully")
else:
    raise ValueError("HF_TOKEN not found in .env file")

## Download SysEngBench Dataset

In [ ]:
import datasets
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("rabell/SysEngBench")

# Access the test split
test_split = dataset["test"]

# Convert the test split to a Pandas DataFrame
df = test_split.to_pandas()

# Check the first few rows of the DataFrame
print(df.head())

## Export Raw Data to CSV

In [ ]:
# Write df to a csv that is utf-8
df.to_csv("sysengbench.csv", index=False, encoding="utf-8")
print(f"Exported {len(df)} questions to sysengbench.csv")

## Data Transformation: Explode INCOSE Categories

Some questions belong to multiple INCOSE Handbook categories. We'll explode these into separate rows for better analysis.

In [ ]:
def explode_incose_handbook_category(df):
    """Explode INCOSE Handbook Category field and extract Category/Sub-Category."""
    # Fill NaN values with empty strings to prevent issues during splitting
    df = df.copy()
    df.loc[:, 'INCOSE Handbook Category'] = df['INCOSE Handbook Category'].fillna('')
    
    # Explode 'INCOSE Handbook Category' by splitting on newline characters
    incose_expanded = df['INCOSE Handbook Category'].str.split('\n').explode()

    # Create a new DataFrame with expanded rows dynamically copying all columns
    incose_expanded_df = df.loc[incose_expanded.index].copy()
    incose_expanded_df['INCOSE Handbook Category'] = incose_expanded.values

    # Split 'INCOSE Handbook Category' into 'INCOSE Category' and 'INCOSE Sub-Category'
    incose_expanded_df[['INCOSE Category', 'INCOSE Sub-Category']] = incose_expanded_df['INCOSE Handbook Category'] \
        .str.extract(r'INCOSEHandbook/([^/]+)/(.+)')

    return incose_expanded_df.reset_index(drop=True)


expanded_df = explode_incose_handbook_category(df)
print(f"Expanded from {len(df)} to {len(expanded_df)} rows")
print(f"\nExpanded DataFrame shape: {expanded_df.shape}")
print(f"\nSample of expanded data:")
print(expanded_df[['Question ID', 'INCOSE Category', 'INCOSE Sub-Category']].head(10))

## Statistical Analysis

In [ ]:
# Calculate total unique MCQs
total_unique_mcqs = expanded_df['Question ID'].nunique()

# Number of MCQs by INCOSE category
mcqs_by_category = expanded_df.groupby('INCOSE Category')['Question ID'].nunique().sort_values(ascending=False)

# Number of MCQs by INCOSE sub-category
mcqs_by_subcategory = expanded_df.groupby('INCOSE Sub-Category')['Question ID'].nunique().sort_values(ascending=False)

# Display results
print(f"Total Unique MCQs: {total_unique_mcqs}")
print(f"\nMCQs by INCOSE Category:")
print(mcqs_by_category)
print(f"\nMCQs by INCOSE Sub-Category:")
print(mcqs_by_subcategory)

## Visualization: Category Distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Group the data by 'INCOSE Category' and count occurrences
category_counts = expanded_df['INCOSE Category'].value_counts()

# Data for the pie chart
categories = category_counts.index
data = category_counts.values

# Colors for the pie chart
colors = ['orange', 'cyan', 'brown', 'grey', 'indigo', 'beige'][:len(categories)]

# Exploding the first slice for better visualization
explode = [0.1 if i == 0 else 0 for i in range(len(categories))]

# Wedge properties
wp = {'linewidth': 1, 'edgecolor': 'black'}

# Custom function to display both percentages and absolute counts
def func(pct, allvalues):
    absolute = int(round(pct / 100. * np.sum(allvalues)))
    return f"{pct:.1f}%\n({absolute:d})"

# Create the pie chart
fig, ax = plt.subplots(figsize=(10, 7))
wedges, texts, autotexts = ax.pie(
    data,
    labels=categories,
    autopct=lambda pct: func(pct, data),
    explode=explode,
    colors=colors,
    startangle=140,
    wedgeprops=wp,
    textprops=dict(color="black")
)

# Adding legend
ax.legend(
    wedges, categories,
    title="INCOSE Categories",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1)
)

# Styling the text
plt.setp(autotexts, size=8, weight="bold")
ax.set_title("INCOSE Handbook Categories Distribution")

# Display the chart
plt.tight_layout()
plt.show()

## Visualization: Sub-Category Distribution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import textwrap

# Group the data by 'INCOSE Category' and 'INCOSE Sub-Category'
subcategory_counts = expanded_df.groupby(['INCOSE Category', 'INCOSE Sub-Category']).size()

# Extract categories and subcategories
categories = subcategory_counts.index.get_level_values(0)
subcategories = subcategory_counts.index.get_level_values(1)
subcategory_values = subcategory_counts.values

# Unique INCOSE categories for color mapping
unique_categories = categories.unique()

# Create a color map for the categories
color_map = {category: plt.cm.tab20c(i / len(unique_categories)) for i, category in enumerate(unique_categories)}

# Assign colors to subcategories based on their parent category
bar_colors = [color_map[category] for category in categories]

# Function to wrap y-axis labels
def wrap_labels(labels, width=25):
    return ['\n'.join(textwrap.wrap(label, width)) for label in labels]

wrapped_subcategories = wrap_labels(subcategories, width=25)

# Horizontal bar chart
plt.figure(figsize=(8, 12), dpi=100)
plt.barh(wrapped_subcategories, subcategory_values, color=bar_colors)

# Add labels and title
plt.xlabel('Number of Questions', fontsize=12, labelpad=10)
plt.ylabel('Subcategory', fontsize=12, labelpad=20)
plt.title('Distribution of MCQs by INCOSE Sub-Category', fontsize=14, weight='bold')

# Add value annotations
for i, v in enumerate(subcategory_values):
    plt.text(v + 2, i, str(v), fontsize=10, va='center')

# Invert y-axis to have the largest bar on top
plt.gca().invert_yaxis()

# Add vertical grid lines
plt.grid(axis='x', linestyle='--', linewidth=0.5, alpha=0.7)

# Explicitly set x-axis limits
plt.xlim(0, max(subcategory_values) * 1.1)

# Create a legend for the categories
legend_handles = [mpatches.Patch(color=color_map[category], label=category) for category in unique_categories]

plt.legend(
    handles=legend_handles,
    title="Categories",
    loc='lower center',
    bbox_to_anchor=(0.5, 1.02),
    fontsize=10,
    ncol=2,
    frameon=False
)

# Adjust layout to fit everything
plt.tight_layout()
plt.show()

## Visualization: Stacked Bar Chart by Category

In [ ]:
import pandas as pd

# Group data by 'INCOSE Category' and 'INCOSE Sub-Category'
category_subcategory_counts = expanded_df.groupby(['INCOSE Category', 'INCOSE Sub-Category']).size().unstack(fill_value=0)

# Stacked bar chart
category_subcategory_counts.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 8),
    colormap='tab20c'
)

# Add labels and title
plt.xlabel('INCOSE Categories', fontsize=12)
plt.ylabel('Number of Questions', fontsize=12)
plt.title('Distribution of MCQs by INCOSE Categories and Sub-Categories', fontsize=14, weight='bold')

plt.legend(title='INCOSE Sub-Categories', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()